# 09 - Modelado con SARIMAX

Ajusta modelos SARIMAX individuales por embalse para la predicción del porcentaje de llenado a 7, 30 y 90 días, en los dos escenarios del diseño experimental.

## Planteamiento

SARIMAX se emplea en su formulación canónica: el modelo se ajusta sobre la serie temporal del porcentaje de llenado de cada embalse y el horizonte de predicción se implementa en la fase de previsión. Las variables exógenas se incorporan con un rezago igual al horizonte, de modo que estén disponibles en el instante en que se genera la predicción.

La estacionalidad anual no se modela mediante la componente estacional del SARIMA, cuyo periodo de 365 días resultaría computacionalmente inviable, sino mediante términos armónicos de Fourier incorporados como regresores deterministas, según la práctica habitual para series diarias con ciclo anual.

Esta formulación difiere de la empleada por los modelos de aprendizaje automático, que trabajan sobre variables objetivo desplazadas. La diferencia no compromete el diseño experimental, ya que la comparación entre escenarios se realiza siempre dentro de un mismo modelo, horizonte y conjunto de embalses.

## Escenarios

- **Base**: hidrología del embalse (caudal de aportación), meteorología (precipitación, acumulados, temperatura y evapotranspiración de referencia) y términos armónicos.
- **Base + calidad**: incorpora las seis variables de calidad del agua.

Ambos escenarios se evalúan sobre el mismo conjunto de 17 embalses, el mismo periodo y la misma muestra de observaciones.

## Validación

Los parámetros se estiman sobre el periodo de entrenamiento (2006-2021) y la evaluación se realiza con origen móvil sobre el periodo reservado (2022-2023): en cada fecha del periodo de test se genera una previsión dinámica a h días sin reestimar el modelo. Entre el fin del entrenamiento y el inicio del test se deja un embargo igual al horizonte.

## Métricas

RMSE, MAE y coeficiente de eficiencia de Nash-Sutcliffe (NSE), este último comparado con el de un modelo de persistencia como referencia.

## 1. Configuración

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

sys.path.append("..")

DIR_PROCESSED = Path("../data/processed")
DIR_FIGURAS = Path("../outputs/figures")
DIR_RESULTADOS = Path("../outputs/tables")
DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)

TRAIN_INI = pd.Timestamp("2006-01-01")
TEST_INI = pd.Timestamp("2022-01-01")
TEST_FIN = pd.Timestamp("2023-12-31")

HORIZONTES = [7, 30, 90]
PARAMETROS = ["amonio_mgl", "conductividad_uscm", "oxigeno_mgl",
              "ph", "temp_agua_c", "turbidez_ntu"]

dataset = pd.read_parquet(DIR_PROCESSED / "dataset_modelado.parquet")
experimentales = sorted(pd.read_parquet(
    DIR_PROCESSED / "embalses_experimento_calidad.parquet")["ID_SAIH"])

# Bloque único de calidad, con prioridad a la estación situada aguas arriba
df = dataset[dataset["ID_SAIH"].isin(experimentales)].copy()
for p in PARAMETROS:
    df[f"cal_{p}"] = df[f"cal_arriba_{p}"].fillna(df[f"cal_abajo_{p}"])

print(f"Observaciones: {len(df):,} | embalses: {len(experimentales)}")
print(f"Periodo: {df['fecha'].min():%Y-%m-%d} a {df['fecha'].max():%Y-%m-%d}")

Observaciones: 111,758 | embalses: 17
Periodo: 2006-01-01 a 2023-12-31


## 2. Definición de los escenarios

Las variables exógenas se agrupan en dos conjuntos que difieren únicamente en la incorporación del bloque de calidad del agua. Los términos armónicos que representan el ciclo anual se incluyen en ambos.

In [2]:
EXOG_BASE = [
    "aportacion_m3s",
    "aemet_precipitacion_mm", "prec_acum7", "prec_acum30",
    "aemet_temp_media_c", "et0_mm",
    "dia_anio_sin", "dia_anio_cos",
]

EXOG_CALIDAD = [f"cal_{p}" for p in PARAMETROS]

ESCENARIOS = {
    "base": EXOG_BASE,
    "base_calidad": EXOG_BASE + EXOG_CALIDAD,
}

for nombre, cols in ESCENARIOS.items():
    print(f"{nombre}: {len(cols)} variables exógenas")
    faltan = [c for c in cols if c not in df.columns]
    if faltan:
        print(f"  AVISO: no están en el dataset -> {faltan}")

base: 8 variables exógenas
base_calidad: 14 variables exógenas


## 3. Preparación de las series

Para cada embalse se construye la serie objetivo y la matriz de variables exógenas sobre una rejilla diaria continua.

Las variables observadas se incorporan con un rezago igual al horizonte de predicción, de modo que la previsión del día t+h se apoye exclusivamente en información disponible en el día t. Los términos armónicos son la excepción, al ser funciones deterministas del calendario conocidas de antemano.

Las variables de calidad tienen medición discontinua (la instrumentación registra por tramos, con paradas de duración variable). Sus ausencias se completan por arrastre hacia delante y hacia atrás dentro del periodo operativo de cada estación, sin extrapolar más allá de sus extremos y de forma independiente en entrenamiento y test para no cruzar esa frontera.

In [3]:
DETERMINISTAS = ["dia_anio_sin", "dia_anio_cos"]

def preparar_serie(df, embalse, exog_cols, horizonte, test_ini=TEST_INI):
    """Serie objetivo y matriz de exógenas de un embalse, sobre rejilla diaria continua.

    Las variables observadas se rezagan el horizonte de predicción. Las de calidad, de
    medición discontinua, se completan por arrastre hacia delante y hacia atrás dentro del
    periodo operativo de la estación, de forma independiente en entrenamiento y test para
    no cruzar esa frontera. Una salvaguarda final cubre huecos residuales de las restantes
    exógenas observadas."""
    g = (df[df["ID_SAIH"] == embalse].sort_values("fecha")
         .set_index("fecha").asfreq("D"))

    y = g["pct_llenado"]
    exog = pd.DataFrame(index=g.index)

    for c in exog_cols:
        s = g[c] if c in DETERMINISTAS else g[c].shift(horizonte)
        if c.startswith("cal_"):
            partes = []
            for tramo in [s[s.index < test_ini], s[s.index >= test_ini]]:
                if tramo.notna().any():
                    ini, fin = tramo.first_valid_index(), tramo.last_valid_index()
                    t = tramo.copy()
                    t.loc[ini:fin] = t.loc[ini:fin].ffill().bfill()
                    partes.append(t)
                else:
                    partes.append(tramo)
            s = pd.concat(partes)
        exog[c] = s

    # Salvaguarda: huecos residuales en exógenas observadas no meteorológicas
    for c in exog.columns:
        if c not in DETERMINISTAS and exog[c].isna().any():
            for tramo_mask in [exog.index < test_ini, exog.index >= test_ini]:
                exog.loc[tramo_mask, c] = exog.loc[tramo_mask, c].ffill().bfill()

    valido = y.notna() & exog.notna().all(axis=1)
    if not valido.any():
        return y.iloc[:0], exog.iloc[:0]
    ini, fin = valido.idxmax(), valido[::-1].idxmax()
    return y.loc[ini:fin], exog.loc[ini:fin]

def indice_valido_comun(df, embalse, horizonte):
    """Fechas con datos completos en el escenario más exigente."""
    y, _ = preparar_serie(df, embalse, ESCENARIOS["base_calidad"], horizonte)
    return y.index

# Comprobación sobre un embalse
y, exog = preparar_serie(df, experimentales[0], ESCENARIOS["base_calidad"], 30)
print(f"Embalse {experimentales[0]}: {len(y):,} observaciones válidas")
print(f"Periodo: {y.index.min():%Y-%m-%d} a {y.index.max():%Y-%m-%d}")
print(f"Exógenas: {exog.shape[1]}")

Embalse E002: 6,574 observaciones válidas
Periodo: 2006-01-01 a 2023-12-31
Exógenas: 14


### Igualdad de la muestra entre escenarios

La rejilla diaria continua y el arrastre acotado garantizan que ambos escenarios se evalúan sobre el mismo conjunto de fechas: las variables de calidad, disponibles para todo el periodo operativo tras el arrastre, no reducen la muestra respecto al escenario base. Ello permite atribuir cualquier diferencia de error exclusivamente a las variables incorporadas y no a un cambio en las observaciones evaluadas.

## 4. Métricas y funciones de evaluación

Se define el conjunto de métricas y las tres funciones de evaluación: SARIMAX con origen móvil, persistencia como referencia, y el cálculo de métricas común a ambas.

In [4]:
def metricas(y_real, y_pred):
    """RMSE, MAE y coeficiente de eficiencia de Nash-Sutcliffe."""
    y_real, y_pred = np.asarray(y_real), np.asarray(y_pred)
    m = ~(np.isnan(y_real) | np.isnan(y_pred))
    y_real, y_pred = y_real[m], y_pred[m]
    if len(y_real) == 0:
        return {"n": 0, "rmse": np.nan, "mae": np.nan, "nse": np.nan}
    err = y_real - y_pred
    sst = ((y_real - y_real.mean()) ** 2).sum()
    return {
        "n": len(y_real),
        "rmse": np.sqrt((err ** 2).mean()),
        "mae": np.abs(err).mean(),
        "nse": 1 - (err ** 2).sum() / sst if sst > 0 else np.nan,
    }

In [5]:
def evaluar_sarimax(y, exog, horizonte, orden=(1, 1, 1), test_ini=TEST_INI):
    """Estima los parámetros sobre el periodo de entrenamiento y evalúa con origen móvil.

    Entre el fin del entrenamiento y el inicio del test se deja un embargo igual al
    horizonte. Los parámetros estimados con entrenamiento se aplican a la serie completa
    sin reestimación. En cada origen t del periodo de test se genera una previsión
    dinámica a h pasos: el modelo emplea las observaciones disponibles hasta t y propaga
    con valores predichos hasta t+h."""
    fin_train = test_ini - pd.Timedelta(days=horizonte)
    y_tr, exog_tr = y[y.index < fin_train], exog[exog.index < fin_train]

    if len(y_tr) < 730 or len(y[y.index >= test_ini]) < horizonte + 30:
        return None, None

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ajuste = SARIMAX(y_tr, exog=exog_tr, order=orden,
                         enforce_stationarity=False,
                         enforce_invertibility=False).fit(disp=False)
        estado = ajuste.apply(y, exog=exog, refit=False)

    idx = y.index
    pos = np.where(idx >= test_ini)[0]
    origenes = pos[pos + horizonte < len(idx)]

    pred = []
    for i in origenes:
        p = estado.get_prediction(start=i + 1, end=i + horizonte, dynamic=True)
        pred.append(p.predicted_mean.iloc[-1])

    return ajuste, pd.DataFrame({
        "fecha": idx[origenes + horizonte],
        "real": y.iloc[origenes + horizonte].values,
        "pred": pred,
    })

In [6]:
def evaluar_persistencia(y, horizonte, test_ini=TEST_INI):
    """Predicción persistente: el valor observado en el origen se mantiene h días.
    Constituye la referencia mínima que el modelo debe superar."""
    idx = y.index
    pos = np.where(idx >= test_ini)[0]
    origenes = pos[pos + horizonte < len(idx)]
    return pd.DataFrame({
        "fecha": idx[origenes + horizonte],
        "real": y.iloc[origenes + horizonte].values,
        "pred": y.iloc[origenes].values,
    })

## 5. Ejecución

Se ajustan y evalúan los modelos para los 17 embalses, los tres horizontes y los dos escenarios, junto con la referencia de persistencia. Los resultados se guardan para su análisis y para la redacción de la memoria.

In [7]:
resultados = []
predicciones = []

for h in HORIZONTES:
    for emb in experimentales:
        # Baseline de persistencia
        y_ref, _ = preparar_serie(df, emb, ESCENARIOS["base"], h)
        res_p = evaluar_persistencia(y_ref, h)
        m = metricas(res_p["real"], res_p["pred"])
        resultados.append({"embalse": emb, "horizonte": h, "escenario": "persistencia", **m})

        # SARIMAX en cada escenario
        for nombre, cols in ESCENARIOS.items():
            try:
                y, exog = preparar_serie(df, emb, cols, h)
                _, res = evaluar_sarimax(y, exog, h)
                if res is None:
                    print(f"  {emb} h{h} {nombre}: datos insuficientes")
                    continue
                m = metricas(res["real"], res["pred"])
                resultados.append({"embalse": emb, "horizonte": h, "escenario": nombre, **m})
                predicciones.append(res.assign(embalse=emb, horizonte=h, escenario=nombre))
            except Exception as e:
                print(f"  ERROR en {emb} h{h} {nombre}: {e}")
                continue
        print(f"  {emb} h{h} listo")

resultados = pd.DataFrame(resultados)
resultados.to_parquet(DIR_RESULTADOS / "sarimax_metricas.parquet", index=False)
pd.concat(predicciones).to_parquet(DIR_RESULTADOS / "sarimax_predicciones.parquet", index=False)

print("\n=== Mediana por horizonte y escenario ===")
print(resultados.groupby(["horizonte", "escenario"])[["rmse", "mae", "nse"]].median().round(3).to_string())

  E002 h7 listo
  E008 h7 listo
  E009 h7 listo
  E011 h7 listo
  E025 h7 listo
  E026 h7 listo
  E027 h7 listo
  E028 h7 listo
  E029 h7 listo
  E030 h7 listo
  E031 h7 listo
  E033 h7 listo
  E07A h7 listo
  E32A h7 listo
  E35A h7 listo
  E570 h7 listo
  E571 h7 listo
  E002 h30 listo
  E008 h30 listo
  E009 h30 listo
  E011 h30 listo
  E025 h30 listo
  E026 h30 listo
  E027 h30 listo
  E028 h30 listo
  E029 h30 listo
  E030 h30 listo
  E031 h30 listo
  E033 h30 listo
  E07A h30 listo
  E32A h30 listo
  E35A h30 listo
  E570 h30 listo
  E571 h30 listo
  E002 h90 listo
  E008 h90 listo
  E009 h90 listo
  E011 h90 listo
  E025 h90 listo
  E026 h90 listo
  E027 h90 listo
  E028 h90 listo
  E029 h90 listo
  E030 h90 listo
  E031 h90 listo
  E033 h90 listo
  E07A h90 listo
  E32A h90 listo
  E35A h90 listo
  E570 h90 listo
  E571 h90 listo

=== Mediana por horizonte y escenario ===
                          rmse    mae    nse
horizonte escenario                         
7         base   

## 6. Resultados

Se examinan las métricas agregadas por horizonte y escenario, el desglose por embalse y la comparación sistemática frente a la persistencia.

In [8]:
tabla = resultados.pivot_table(index="embalse", columns=["horizonte", "escenario"],
                               values="nse").round(3)
print(tabla[30].to_string())

# Cuántos embalses supera SARIMAX a la persistencia
for h in HORIZONTES:
    r = resultados[resultados["horizonte"] == h].pivot_table(
        index="embalse", columns="escenario", values="nse")
    mejora = (r["base"] > r["persistencia"]).sum()
    print(f"h={h}: SARIMAX base supera a persistencia en {mejora}/17 embalses")

escenario   base  base_calidad  persistencia
embalse                                     
E002      -0.459        -0.468        -0.500
E008      -0.311        -0.310        -0.840
E009      -0.278        -0.271        -0.370
E011      -0.075        -0.073        -0.113
E025      -0.037        -0.039        -0.059
E026       0.391         0.388         0.308
E027      -0.605        -0.608        -0.410
E028       0.879         0.879         0.705
E029      -1.240        -1.237        -0.958
E030      -0.275        -0.272        -0.755
E031      -0.047        -0.053        -0.466
E033      -0.314        -0.315        -0.803
E07A       0.819         0.819         0.621
E32A      -0.893        -0.892        -0.780
E35A       0.451         0.452         0.360
E570       0.653         0.655         0.658
E571      -0.407        -0.400        -1.105
h=7: SARIMAX base supera a persistencia en 14/17 embalses
h=30: SARIMAX base supera a persistencia en 13/17 embalses
h=90: SARIMAX base supera a 

## 7. Conclusiones

Los modelos SARIMAX se ajustaron correctamente para los 17 embalses en los tres horizontes y los dos escenarios, junto con la referencia de persistencia.

**Rendimiento por horizonte.** A 7 días el modelo alcanza un NSE mediano de 0,639 y supera a la persistencia (0,634). A 30 y 90 días el NSE mediano es negativo (−0,275 y −0,433 respectivamente), pero el modelo mantiene ventaja sobre la persistencia, cuya mediana desciende a −0,410 y −0,891. El deterioro con el horizonte es esperable, dado que la previsión a mayor plazo depende cada vez menos del estado actual de la serie.

**Comparación con la persistencia.** SARIMAX supera a la predicción persistente en 14, 13 y 14 de los 17 embalses a 7, 30 y 90 días respectivamente. Este es el resultado más relevante del modelo: un NSE negativo no indica fracaso, sino que la serie es difícil de predecir mejor que su propia media; la comparación con la persistencia demuestra que el modelo aporta capacidad predictiva sobre la referencia natural del problema en la gran mayoría de embalses.

**Dependencia del régimen de explotación.** El rendimiento no se explica por la magnitud de la variabilidad sino por su estructura temporal. Los embalses de regulación estacional con dinámica predecible obtienen los mejores resultados (Vilasouto 0,879, Bárcena 0,819, Santiago 0,653, Conchas 0,451, Edrada 0,391 a 30 días), mientras que los de dinámica rápida o muy plana resultan poco predecibles a horizontes largos. El caso de Santiago (E570), el embalse de menor variabilidad, es ilustrativo: es el único en que la persistencia iguala al modelo, al ser su nivel actual un predictor casi perfecto a 30 días.

**Aporte de las variables de calidad.** Los escenarios base y base+calidad arrojan métricas prácticamente idénticas en los 17 embalses, con diferencias en la tercera cifra decimal. En la formulación lineal de SARIMAX, el bloque de calidad del agua no aporta capacidad predictiva sobre el escenario base, en coherencia con el análisis de correlaciones del apartado anterior, donde ninguna variable de calidad mostró asociación apreciable con la variación futura del llenado.

**Continuidad.** Queda abierto si los modelos de aprendizaje automático, capaces de capturar relaciones no lineales e interacciones entre variables, encontrarán en el bloque de calidad un aporte que la formulación lineal no detecta. Su evaluación constituye el objeto de la siguiente fase del trabajo.